In [1]:
import numpy as np

def longest_failure_run(N, p):
    """
    N : int - number of trials
    p : float - probability of success
    Returns:
        f_expected : approximate expected longest run of failures
        probs : dictionary of run length -> probability
    """
    q = 1 - p
    
    # Approximate expected longest run
    f_expected = np.log(N) / np.log(1/q)
    
    # Compute probability distribution for runs
    # Only consider f around expected value for efficiency
    f_min = max(1, int(f_expected) - 5)
    f_max = int(f_expected) + 5
    
    probs = {}
    for f in range(f_min, f_max + 1):
        # P(longest run = f) ≈ exp(-N*q^(f+1)) - exp(-N*q^f)
        prob = np.exp(-N * q**(f+1)) - np.exp(-N * q**f)
        probs[f] = prob
    
    return f_expected, probs

In [7]:
N = 2000
p = 1/3.6 * 0.99

f_expected, probs = longest_failure_run(N, p)
print("The expected longest consecutive failures: {}".format(f_expected))

for f, p in probs.items():
    print("The longest consecutive failurss: {}, probability: {:.5f}".format(f, p))

The expected longest consecutive failures: 23.635850488858853
The longest consecutive failurss: 18, probability: 0.00960
The longest consecutive failurss: 19, probability: 0.02819
The longest consecutive failurss: 20, probability: 0.05692
The longest consecutive failurss: 21, probability: 0.08721
The longest consecutive failurss: 22, probability: 0.10910
The longest consecutive failurss: 23, probability: 0.11766
The longest consecutive failurss: 24, probability: 0.11386
The longest consecutive failurss: 25, probability: 0.10182
The longest consecutive failurss: 26, probability: 0.08596
The longest consecutive failurss: 27, probability: 0.06961
The longest consecutive failurss: 28, probability: 0.05468


In [8]:
N = 10000
p = 1/3.6 * 0.99

f_expected, probs = longest_failure_run(N, p)
print("The expected longest consecutive failures: {}".format(f_expected))

for f, p in probs.items():
    print("The longest consecutive failurss: {}, probability: {:.5f}".format(f, p))

The expected longest consecutive failures: 28.64057645026598
The longest consecutive failurss: 23, probability: 0.00954
The longest consecutive failurss: 24, probability: 0.02807
The longest consecutive failurss: 25, probability: 0.05677
The longest consecutive failurss: 26, probability: 0.08708
The longest consecutive failurss: 27, probability: 0.10903
The longest consecutive failurss: 28, probability: 0.11765
The longest consecutive failurss: 29, probability: 0.11390
The longest consecutive failurss: 30, probability: 0.10189
The longest consecutive failurss: 31, probability: 0.08604
The longest consecutive failurss: 32, probability: 0.06969
The longest consecutive failurss: 33, probability: 0.05475


In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

class ChunkedMultiheadAttention(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        nHeads: int,
        device: Optional[str] = None,
        num_chunks: int = 1,
        dropout: float = 0.0,
        bias: bool = True
    ):
        super().__init__()
        assert embed_dim % nHeads == 0, "embed_dim must be divisible by nHeads"
        self.embed_dim = embed_dim
        self.nHeads = nHeads
        self.head_dim = embed_dim // nHeads
        self.dropout = dropout

        if device is None:
            device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        self.device = torch.device(device)

        # Partition nHeads into chunks
        base = nHeads // num_chunks
        rem = nHeads % num_chunks
        self.heads_per_chunk = [base + (1 if i < rem else 0) for i in range(num_chunks)]

        # qkv projections (on GPU)
        self.qkv_linears = nn.ModuleList()
        for nHeads in self.heads_per_chunk:
            d_local = nHeads * self.head_dim
            lin = nn.Linear(embed_dim, 3 * d_local, bias=bias).to(self.device)
            self.qkv_linears.append(lin)

        # Final out projection (on CPU!)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias).to("cpu")

        self.attn_dropout = nn.Dropout(dropout)
        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None):
        B, S, E = x.shape
        assert E == self.embed_dim

        x = x.to(self.device)
        per_chunk_outputs = []

        for idx, nHeads in enumerate(self.heads_per_chunk):
            d_local = nHeads * self.head_dim
            qkv = self.qkv_linears[idx](x)  # (B, S, 3*d_local)
            q, k, v = torch.chunk(qkv, 3, dim=-1)

            q = q.view(B, S, nHeads, self.head_dim).transpose(1, 2)
            k = k.view(B, S, nHeads, self.head_dim).transpose(1, 2)
            v = v.view(B, S, nHeads, self.head_dim).transpose(1, 2)

            attn_logits = torch.matmul(q, k.transpose(-2, -1)) * self.scale

            if mask is not None:
                mask_dev = mask.to(self.device)
                if mask_dev.dtype == torch.bool:
                    attn_logits = attn_logits.masked_fill(~mask_dev.unsqueeze(1), float('-inf'))
                else:
                    attn_logits = attn_logits + mask_dev.unsqueeze(1)

            attn = F.softmax(attn_logits, dim=-1)
            attn = self.attn_dropout(attn)

            out = torch.matmul(attn, v)  # (B, nHeads, S, head_dim)
            out = out.transpose(1, 2).contiguous().view(B, S, d_local)
            per_chunk_outputs.append(out)

        # Concatenate on GPU
        concat = torch.cat(per_chunk_outputs, dim=-1)  # (B, S, embed_dim)

        # Move concat to CPU for out_proj
        concat_cpu = concat.to("cpu")
        out_cpu = self.out_proj(concat_cpu)

        # If you want the result back on GPU for downstream layers
        out = out_cpu.to(self.device)

        return out


In [11]:
patch_size = 16
device = 'cpu' #'cuda:0' # if torch.cuda.is_available() else 'cpu'
mha = ChunkedMultiheadAttention(embed_dim=1024, nHeads=64, device=device, num_chunks=4)

x = torch.randn(24, round(720/patch_size * 1440/patch_size), round(patch_size*patch_size*4))  # (B, S, E)
out = mha(x)  # (16, 128, 512)
print(out.shape)

RuntimeError: [enforce fail at alloc_cpu.cpp:114] data. DefaultCPUAllocator: not enough memory: you tried to allocate 25194240000 bytes.